In [0]:
%run /Workspace/Users/kaustuvdev8123@outlook.com/utilClass/Encryption


In [0]:
from delta.tables import DeltaTable
import time




base_path = "abfss://data@stdevwesteuropewrmp.dfs.core.windows.net"

read_path = base_path + "/silver/hotel_weather_processed"

gold_write_path = base_path + "/gold/hotel_weather_metrics"

schema_location_path = base_path + "/metadata/gold/schema"



encrypted_silver = spark.readStream\
    .format("delta")\
    .option('inferSchema', True)\
    .load(read_path)

columns_to_be_decrypted = ("address", "city", "country", "name")

decrypted_silver = Encrypter().decrypt_columns(
    encrypted_silver, *columns_to_be_decrypted
)



Silver table is ready. Starting stream.
('address', 'city', 'country', 'name')


In [0]:
import pyspark.sql.functions as F
import os
import uuid
metrics_df = decrypted_silver.groupBy("wthr_date", "city", "country").agg(
    F.approx_count_distinct("id").alias("hotel_count"),
    F.avg("avg_tmpr_c").alias("avg_tmpr_c"),
    F.max("avg_tmpr_c").alias("max_tmpr_c"),
    F.min("avg_tmpr_c").alias("min_tmpr_c")
)
metrics_df = metrics_df.withColumn(
    "tmpr_diff",
    F.col("max_tmpr_c") - F.col("min_tmpr_c")
)

# display(metrics_df, checkpointLocation = os.getcwd() + f"/tmp/checkpoint/gold_{uuid.uuid4()}")
metrics_df.writeStream\
    .format("delta")\
    .option("checkpointLocation", f"{base_path}/checkpoint/gold_checkpoint")\
    .outputMode("complete")\
    .start(gold_write_path)